# Graph Fraud Detection — Model Training (ML Job + Tuner API)

Demonstrates two Snowflake-native training paths on the graph-fraud training dataset:

| Step | Method | Description |
|------|--------|-------------|
| 1 | **ML Job** (`submit_directory`) | Baseline XGBoost — runs `source/train.py` on SPCS compute pool |
| 2 | **Tuner API** (BayesOpt) | Distributed HPO in Container Runtime or via `@remote` ML Job |
| 3 | **Model Registry** | Register best model as `FRAUD_RING_MODEL` V1 (prod) and V2 (candidate) |

Both paths use the point-in-time correct training data from `FRAUD_TRAINING_DATA`
(generated in `01_features.ipynb` via `generate_training_set`).

**Run in Container Runtime** for Tuner API support.

In [1]:
import os
import pickle
import time
from datetime import datetime

import snowflake.connector
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.model import type_hints

try:
    from snowflake.ml.modeling.tune import Tuner
    HAS_TUNER = True
except ImportError:
    HAS_TUNER = False

CONNECTION_NAME  = 'parker_demo'
DATABASE         = 'GRAPH_FRAUD_DEMO'
SCHEMA           = 'MODELS'
WAREHOUSE        = 'GRAPH_FRAUD_WH'
COMPUTE_POOL     = 'ML_TRAINING_POOL'
MODEL_NAME       = 'FRAUD_RING_MODEL'
EXPERIMENT_NAME  = 'GRAPH_FRAUD_EXPERIMENT'
TRAINING_TABLE   = f'{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA'
MODEL_STAGE      = f'@{DATABASE}.{SCHEMA}.ML_MODELS'

FEATURE_COLS = [
    'EFFECTIVE_COMMUNITY_SIZE',
    'DEGREE',
    'SHARED_ADDRESS_COUNT',
    'SHARED_IP_COUNT',
    'AVG_COMMUNITY_REFUND',
    'COMMUNITY_REFUND_STDDEV',
    'ADDR_N_FILERS',
    'ADDR_AVG_REFUND',
    'IP_N_FILERS',
    'IP_AVG_REFUND',
    'SUBMISSION_HOUR',
    'IS_HIGH_CLAIM',
    'IS_ODD_HOUR',
    'CLAIM_AMOUNT',
    'HAS_DIRECT_GRAPH',
]

print(f'Tuner API available: {HAS_TUNER}')

# Load PAT from Snowflake config
import tomllib, pathlib
_sf_config = tomllib.load(open(pathlib.Path.home() / '.snowflake' / 'config.toml', 'rb'))
os.environ.setdefault('SNOWFLAKE_PAT', _sf_config['connections']['parker_demo']['password'])
print(f'SNOWFLAKE_PAT loaded ({len(os.environ["SNOWFLAKE_PAT"])} chars)')


Tuner API available: False
SNOWFLAKE_PAT loaded (218 chars)


In [2]:
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    conn = snowflake.connector.connect(connection_name=CONNECTION_NAME)
    session = Session.builder.configs({'connection': conn}).create()

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'USE SCHEMA {DATABASE}.{SCHEMA}').collect()
session.sql(f'USE WAREHOUSE {WAREHOUSE}').collect()
print(f'Connected: {session.get_current_account()}')

Connected: "SFSENORTHAMERICA-PERICKSON_AWS1"


## Step 1: Baseline Training via ML Job

Submits `source/train.py` as a job on an SPCS compute pool using
`submit_directory()`. The job runs in isolation — the entire `source/`
directory is uploaded to a stage and executed on SPCS compute.

The 15 graph-community features capture:
- **Direct Leiden features** (community size, degree, shared-attribute counts)
- **Address-inherited** context (applicants at the same address)
- **IP-inherited** context (applicants sharing the same IP)
- **Velocity** signals (filing hour, high claim, odd hour)
- **Request-time** features (claim amount, has-direct-graph flag)

In [3]:
session.sql("DROP GATEWAY IF EXISTS GRAPH_FRAUD_DEMO.MODELS.GRAPH_FRAUD_GATEWAY").collect()
session.sql("DROP SERVICE IF EXISTS GRAPH_FRAUD_DEMO.MODELS.GRAPH_FRAUD_SVC_V1").collect()
session.sql("DROP SERVICE IF EXISTS GRAPH_FRAUD_DEMO.MODELS.GRAPH_FRAUD_SVC_V2").collect()
print("Services + gateway dropped")

session.sql("DROP MODEL IF EXISTS GRAPH_FRAUD_DEMO.MODELS.FRAUD_RING_MODEL").collect()
print("Model dropped")

Services + gateway dropped
Model dropped


In [4]:
from snowflake.ml.jobs import submit_directory

session.sql(f'''
CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL}
  MIN_NODES = 1 MAX_NODES = 3
  INSTANCE_FAMILY = CPU_X64_S
''').collect()
print(f'Compute pool: {COMPUTE_POOL}')

job = submit_directory(
    dir_path='../source',
    entrypoint='train.py',
    compute_pool=COMPUTE_POOL,
    stage_name=f'{DATABASE}.{SCHEMA}.ML_JOB_STAGE',
    env_vars={
        'DATABASE':        DATABASE,
        'SCHEMA':          SCHEMA,
        'EXPERIMENT_NAME': EXPERIMENT_NAME,
        'REGISTER_MODEL':  'true',
        'VERSION_NAME':    'V1',
        'MODEL_STAGE':     MODEL_STAGE,
    },
)
print(f'Job submitted: {job.id}')
print(f'Status      : {job.status}')

Compute pool: ML_TRAINING_POOL
Job submitted: GRAPH_FRAUD_DEMO.MODELS.TRAIN_FBC2C2F9_Z63W201ZFIQD
Status      : PENDING


In [5]:
print('Waiting for ML Job to complete...')
for i in range(60):
    status = job.status
    print(f'  [{i:02d}] {status}')
    if status in ('DONE', 'FAILED', 'CANCELLED'):
        break
    time.sleep(30)

if job.status == 'DONE':
    print(f'\nJob completed successfully')
    try:
        print(job.get_logs())
    except Exception as e:
        print(f'Logs: {e}')
else:
    print(f'\nJob ended with status: {job.status}')
    print(job.get_logs())

Waiting for ML Job to complete...
  [00] PENDING
  [01] RUNNING
  [02] RUNNING
  [03] RUNNING
  [04] RUNNING
  [05] PENDING
  [06] DONE

Job completed successfully
2026-10-15 17:04:40,937	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 3221225472 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.18gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-10-15 17:04:44,873 - INFO - generated new fontManager
2026-10-15 17:04:28,126	INFO usage_lib.py:448 -- Usage stats collection is disabled.
2026-10-15 17:04:28,126	INFO scripts.py:940 -- Local node IP: 10.244.8.202
2026-10-15 17:04:45,170	SUCC scripts.py:979 -- --------------------
2026-10-15 17:04:45,170	SUCC scripts.py:980 -- Ray ru

### Register Baseline Model as V1 (Production)

Download the trained model artifact from stage, register to Model Registry as `V1`.
This becomes the **production model** the Gateway initially routes 100% traffic to.

In [6]:
# Verify model was registered by the ML Job (no local loading needed)
reg = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
model_ref = reg.get_model(MODEL_NAME)
mv = model_ref.version('V1')

print(f'Model: {MODEL_NAME}')
print(f'Version: V1')
print(f'Metrics: {mv.show_metrics()}')
print(f'Comment: {mv.comment}')
print(f'\nV1 registered successfully — no local artifacts needed.')


Model: FRAUD_RING_MODEL
Version: V1
Metrics: {'auc': 0.9935656640048078, 'recall': 0.979381443298969, 'precision': 0.3007915567282322, 'f1': 0.4602341542188131}
Comment: Graph fraud model — AUC: 0.9936

V1 registered successfully — no local artifacts needed.


## Step 2: Hyperparameter Optimization via Tuner API

Uses **Bayesian Optimization** (UCB acquisition function) to search for the best
XGBoost hyperparameters — 10 trials on a 5-parameter search space.

The Tuner runs in **Container Runtime** if available (in-process on SPCS GPU/CPU),
otherwise falls back to `submit_directory` with manually tuned params.

In [7]:
SEARCH_SPACE = {
    'n_estimators':    (50, 400),
    'max_depth':       (3, 10),
    'learning_rate':   (0.01, 0.3),
    'scale_pos_weight': (10, 80),
    'min_child_weight': (1, 20),
}
NUM_TRIALS = 10

print('BayesOpt search space:')
for k, v in SEARCH_SPACE.items():
    print(f'  {k}: {v}')

BayesOpt search space:
  n_estimators: (50, 400)
  max_depth: (3, 10)
  learning_rate: (0.01, 0.3)
  scale_pos_weight: (10, 80)
  min_child_weight: (1, 20)


In [8]:
def run_hpo(session, train_table, search_space_bounds, num_trials):
    """Run distributed HPO — Container Runtime or @remote ML Job fallback."""
    from snowflake.ml.modeling.tune import Tuner, TunerConfig, get_tuner_context
    from snowflake.ml.modeling.tune.search import BayesOpt
    from snowflake.ml.modeling import tune
    from snowflake.ml.data.data_connector import DataConnector
    import xgboost as xgb
    from sklearn.metrics import roc_auc_score

    fraud_df = session.table(train_table)
    train_df = fraud_df.sample(frac=0.8)
    test_df  = fraud_df.subtract(train_df)

    dataset_map = {
        'train': DataConnector.from_dataframe(train_df),
        'test':  DataConnector.from_dataframe(test_df),
    }

    search_space = {k: tune.uniform(v[0], v[1]) for k, v in search_space_bounds.items()}

    def train_fn():
        ctx = get_tuner_context()
        config = ctx.get_hyper_params()
        dm = ctx.get_dataset_map()
        train_pd = dm['train'].to_pandas()
        test_pd  = dm['test'].to_pandas()

        available = [c for c in FEATURE_COLS if c in train_pd.columns]
        X_train = train_pd[available].fillna(0)
        y_train = train_pd['IS_FRAUD'].fillna(0)
        X_test  = test_pd[available].fillna(0)
        y_test  = test_pd['IS_FRAUD'].fillna(0)

        m = xgb.XGBClassifier(
            n_estimators=int(config['n_estimators']),
            max_depth=int(config['max_depth']),
            learning_rate=config['learning_rate'],
            scale_pos_weight=config['scale_pos_weight'],
            min_child_weight=config['min_child_weight'],
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            eval_metric='logloss',
        )
        m.fit(X_train, y_train)
        y_proba = m.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
        ctx.report(metrics={'auc': auc}, model=m)

    tuner_config = TunerConfig(
        metric='auc',
        mode='max',
        search_alg=BayesOpt(utility_kwargs={'kind': 'ucb', 'kappa': 2.5}),
        num_trials=num_trials,
        max_concurrent_trials=1,
    )

    tuner = Tuner(
        train_func=train_fn,
        search_space=search_space,
        tuner_config=tuner_config,
    )

    return tuner.run(dataset_map=dataset_map)

In [9]:
if HAS_TUNER:
    print(f'Running {NUM_TRIALS}-trial BayesOpt HPO in Container Runtime...')
    hpo_results = run_hpo(session, TRAINING_TABLE, SEARCH_SPACE, NUM_TRIALS)
else:
    print('Tuner API not available locally — submitting as @remote ML Job...')
    from snowflake.ml.jobs import submit_directory
    hpo_job = submit_directory(
        dir_path='../source',
        entrypoint='train.py',
        compute_pool=COMPUTE_POOL,
        stage_name=f'{DATABASE}.{SCHEMA}.ML_JOB_STAGE',
        env_vars={
            'DATABASE':        DATABASE,
            'SCHEMA':          SCHEMA,
            'EXPERIMENT_NAME': EXPERIMENT_NAME,
            'MODEL_STAGE':     MODEL_STAGE,
            'REGISTER_MODEL':  'true',
            'VERSION_NAME':    'V2',
            'N_ESTIMATORS':    '300',
            'MAX_DEPTH':       '6',
            'LEARNING_RATE':   '0.03',
        },
    )
    print(f'HPO job submitted: {hpo_job.id}')
    for i in range(60):
        s = hpo_job.status
        print(f'  [{i:02d}] {s}')
        if s in ('DONE', 'FAILED', 'CANCELLED'):
            break
        time.sleep(30)
    print(hpo_job.get_logs())

Tuner API not available locally — submitting as @remote ML Job...
HPO job submitted: GRAPH_FRAUD_DEMO.MODELS.TRAIN_DA6DC90B_1J77554HOUJUD
  [00] PENDING
  [01] RUNNING
  [02] RUNNING
  [03] RUNNING
  [04] PENDING
  [05] PENDING
  [06] DONE
2026-10-15 17:07:40,828	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 3221225472 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.18gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-10-15 17:07:43,117 - INFO - generated new fontManager
2026-10-15 17:07:43,580 - INFO - Snowflake Connector for Python Version: 4.7.3, Python Version: 3.11.14, Platform: Linux-5.15.221-14.2026092810g1270310+snow+aws+5.15+amd64.x86_64-x86_64-with-glibc2.39
2026

### Extract Best Params & Register as V2 (Candidate)

In [10]:
if HAS_TUNER:
    from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score
    from sklearn.model_selection import train_test_split

    best_row    = hpo_results.best_result.iloc[0]
    best_params = {
        'n_estimators':     int(best_row['config/n_estimators']),
        'max_depth':        int(best_row['config/max_depth']),
        'learning_rate':    float(best_row['config/learning_rate']),
        'scale_pos_weight': float(best_row['config/scale_pos_weight']),
        'min_child_weight': float(best_row['config/min_child_weight']),
    }
    best_auc = float(best_row['auc'])
    print(f'Best params : {best_params}')
    print(f'Best AUC    : {best_auc:.4f}')

    tuned_model = hpo_results.best_model

    run_name = f'xgb_graph_fraud_tuned_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    exp = ExperimentTracking(session=session, database_name=DATABASE, schema_name=SCHEMA)
    exp.set_experiment(EXPERIMENT_NAME)
    with exp.start_run(run_name):
        exp.log_params({**{k: str(round(v, 4)) for k, v in best_params.items()}, 'run_type': 'graph_fraud_tuner'})
        exp.log_metrics({'test_auc_roc': best_auc})
    print(f'Logged run: {run_name}')

    train_pdf = session.table(TRAINING_TABLE).to_pandas()
    model_features = tuned_model.get_booster().feature_names
    X = train_pdf[model_features].fillna(0)
    y = train_pdf['IS_FRAUD'].fillna(0)
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    y_proba = tuned_model.predict_proba(X_test)[:, 1]
    y_pred  = tuned_model.predict(X_test)
    tuned_metrics = {
        'auc':       float(roc_auc_score(y_test, y_proba)),
        'recall':    float(recall_score(y_test, y_pred, zero_division=0)),
        'precision': float(precision_score(y_test, y_pred, zero_division=0)),
        'f1':        float(f1_score(y_test, y_pred, zero_division=0)),
    }
    print(f'Test metrics: {tuned_metrics}')

    sample_df = session.create_dataframe(X_test.head(100))
    reg.log_model(
        model=tuned_model,
        model_name=MODEL_NAME,
        version_name='V2',
        sample_input_data=sample_df,
        metrics=tuned_metrics,
        comment=f'Tuned graph fraud model (BayesOpt) — AUC: {tuned_metrics["auc"]:.4f}',
        target_platforms=[type_hints.TargetPlatform.SNOWPARK_CONTAINER_SERVICES],
    )
    print(f'Registered: {MODEL_NAME}:V2 (candidate — AUC {tuned_metrics["auc"]:.4f})')

## Registry Summary

In [11]:
model_ref = reg.get_model(MODEL_NAME)
print(f'Model: {MODEL_NAME}')
for mv in model_ref.versions():
    m = mv.show_metrics()
    auc = m.get('auc', None)
    auc_str = f'{auc:.4f}' if isinstance(auc, (int, float)) else 'N/A'
    print(f'  {mv.version_name}  AUC={auc_str}  comment="{mv.comment}"')
print('\nProceed to 03_deploy_gateway.ipynb')

Model: FRAUD_RING_MODEL
  V1  AUC=0.9936  comment="Graph fraud model — AUC: 0.9936"
  V2  AUC=0.9935  comment="Graph fraud model — AUC: 0.9935"

Proceed to 03_deploy_gateway.ipynb
